## 1. Imports, Data Recovery & Split Setup

### 🎯 Objectives:
1. Load the dataset and import essential Scikit-Learn and XGBoost modeling libraries.
2. Re-create our stratified 60% Train, 20% Validation, 20% Test partitions (`random_state=42`).
3. Re-define the standardized `evaluate_model` helper function from Stage 3.

### 💡 Conceptual Explanation & Intuition:
- **Reproducibility & Data Lineage:** To ensure fair model comparisons across notebooks, we preserve the exact same `random_state=42` and stratification strategy established in Section 3. Every model in this notebook will be trained on the identical 60% training partition and evaluated against the identical 20% validation partition.
- **Metric Consistency:** Our primary benchmark metric remains **Macro F1-Score**, which evaluates the model performance equally across all 5 stress levels.


In [1]:
# --- Predefined imports, paths, and configurations ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

# Styling
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (8, 5)
pd.set_option("display.max_columns", None)

# 1. Load and clean column names
DATA_PATH = "../data/mental_health_analysis.csv"
df = pd.read_csv(DATA_PATH)
df.columns = df.columns.str.lower().str.replace(" ", "_")

# 2. Define features, target, and 0-indexed adjustment
target_col = "survey_stress_score"
dropped_cols = ["user_id", "wearable_stress_score", target_col]
numerical_cols = ["age", "social_media_hours", "exercise_hours", "sleep_hours", "screen_time_hours"]
categorical_cols = ["gender", "support_system", "academic_performance"]

X = df.drop(columns=dropped_cols)
y = df[target_col] - 1  # 0-indexed (0, 1, 2, 3, 4)

# 3. Stratified 60% Train / 20% Val / 20% Test split
RANDOM_STATE = 42
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.25, stratify=y_train_full, random_state=RANDOM_STATE
)

# 4. Standardized evaluation helper function
target_names = ["Level 1", "Level 2", "Level 3", "Level 4", "Level 5"]
display_labels = ["Lvl 1", "Lvl 2", "Lvl 3", "Lvl 4", "Lvl 5"]

def evaluate_model(y_true, y_pred, model_name="Model", display_cm=True):
    """
    Computes accuracy, macro F1, weighted F1, and displays confusion matrix.
    """
    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    weighted_f1 = f1_score(y_true, y_pred, average="weighted")
    
    print(f"=== {model_name} Evaluation ===")
    print(f"Accuracy:     {acc:.4f}")
    print(f"Macro F1:     {macro_f1:.4f} (Primary Metric)")
    print(f"Weighted F1:  {weighted_f1:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=target_names))
    
    if display_cm:
        fig, ax = plt.subplots(figsize=(5, 4))
        ConfusionMatrixDisplay.from_predictions(
            y_true, y_pred,
            display_labels=display_labels,
            cmap="Blues",
            ax=ax
        )
        ax.set_title(f"Confusion Matrix — {model_name}")
        plt.grid(False)
        plt.show()
        
    return {
        "model": model_name,
        "accuracy": round(acc, 4),
        "macro_f1": round(macro_f1, 4),
        "weighted_f1": round(weighted_f1, 4)
    }

print(f"Dataset ready: Train={X_train.shape}, Val={X_val.shape}, Test={X_test.shape}")


Dataset ready: Train=(3000, 8), Val=(1000, 8), Test=(1000, 8)


---
## 2. Feature Preprocessing Pipeline (ColumnTransformer)

### 🎯 Objectives:
1. Construct a Scikit-Learn `ColumnTransformer` that applies:
   - **One-Hot Encoding** to categorical columns (`gender`, `support_system`, `academic_performance`).
   - **Standard Scaling** to numerical features (`age`, `sleep_hours`, `screen_time_hours`, etc.).
2. Fit the preprocessor on `X_train` and inspect the transformed feature matrix dimensions.

### 💡 Conceptual Explanation & Intuition:
- **Why Preprocess?** Machine learning models operate on mathematical distance, dot products, and matrix operations. They cannot natively parse text strings (`"Moderate"`, `"Excellent"`), and features with large numerical magnitudes can unintentionally dominate optimization gradients.
- **One-Hot Encoding (`OneHotEncoder`):** Converts each categorical variable with $C$ unique categories into $C$ binary indicator columns ($0$ or $1$).
  - Setting `handle_unknown=ignore` is critical for production: if an unseen category appears during inference, the encoder assigns zeros across all indicators rather than crashing with an exception.
- **Standard Scaling (`StandardScaler`):** Standardizes numerical values by subtracting the mean and dividing by standard deviation:
  $z = \frac{x - \mu}{\sigma}$
  This centers features around $\mu=0$ with unit variance $\sigma=1$.
  - *Why this matters for linear models:* Without scaling, gradient descent updates and $\ell_2$ regularization penalties in Logistic Regression disproportionately penalize features with larger scales (e.g. `screen_time_hours` vs `exercise_hours`).
- **`ColumnTransformer`:** Packages multiple column-specific transformations into a single unified step, eliminating manual dataframe slicing and preventing train-test data leakage.


In [2]:
# --- Predefined transformer objects and column groupings ---
categorical_transformer = OneHotEncoder(handle_unknown="ignore")
numerical_transformer = StandardScaler()

# YOUR CODE HERE:
# 1. Create a ColumnTransformer named preprocessor combining numerical and categorical transformers
preprocessor = None  # TODO: ColumnTransformer(transformers=[...])

# 2. Fit the preprocessor on X_train and transform X_train into X_train_proc
# X_train_proc = preprocessor.fit_transform(X_train)

# 3. Test invocation: print original vs transformed shape
# print("Original X_train shape:", X_train.shape)
# print("Transformed X_train shape:", X_train_proc.shape)


---
## 3. Linear Baseline Model: Logistic Regression

### 🎯 Objectives:
1. Construct an end-to-end `Pipeline` combining `preprocessor` and `LogisticRegression(max_iter=1000, random_state=42)`.
2. Train the baseline model on `(X_train, y_train)` and generate predictions on `X_val`.
3. Evaluate baseline metrics using `evaluate_model` and store results in a benchmark summary list.

### 💡 Conceptual Explanation & Intuition:
- **What is Multi-Class Logistic Regression (Softmax Regression)?**
  For $K$ classes (here $K=5$), the model computes a linear score for each class:
  $z_k = \mathbf{w}_k^T \mathbf{x} + b_k$
  These linear scores are converted into normalized probabilities using the **Softmax function**:
  $P(y = k \mid \mathbf{x}) = \frac{e^{z_k}}{\sum_{j=1}^K e^{z_j}}$
  The predicted class is the one with the highest probability: $\hat{y} = \arg\max_k P(y=k \mid \mathbf{x})$.
- **Why Start with a Baseline?** A baseline provides the ground truth reference point. Complex algorithms (Decision Trees, Random Forests, XGBoost) involve higher computational cost and complexity; they are only justified if they measurably beat this simple linear baseline.


In [ ]:
# --- Predefined model hyperparameters and experiment log ---
LR_MAX_ITER = 1000
LR_RANDOM_STATE = 42
experiments_summary = []

# YOUR CODE HERE:
# 1. Build a Pipeline combining preprocessor and LogisticRegression
# lr_pipeline = Pipeline(steps=[
#     (preprocessor, preprocessor),
#     (classifier, LogisticRegression(max_iter=LR_MAX_ITER, random_state=LR_RANDOM_STATE))
# ])

# 2. Fit the pipeline on (X_train, y_train) and predict on X_val
# lr_pipeline.fit(X_train, y_train)
# y_val_pred_lr = lr_pipeline.predict(X_val)

# 3. Evaluate the model using evaluate_model helper
# lr_results = evaluate_model(y_val, y_val_pred_lr, model_name="Logistic Regression (Baseline)")
# experiments_summary.append(lr_results)


---
## 4. Tree-Based Model: Decision Tree Classifier

### 🎯 Objectives:
1. Build and train a `DecisionTreeClassifier` within our preprocessing pipeline.
2. Evaluate non-linear decision boundary performance on `X_val`.
3. Compare decision tree validation scores against the linear baseline.

### 💡 Conceptual Explanation & Intuition:
- **What is a Decision Tree?** A decision tree partitions the feature space into axis-aligned rectangular regions by recursively asking binary if-else questions (e.g. `sleep_hours <= 6.5`, `screen_time_hours > 8.0`).
- **Splitting Criteria (Gini Impurity):** At each node, the algorithm chooses the feature and threshold that maximizes the reduction in impurity (making the child subsets as homogeneous as possible):
  $I_G(p) = 1 - \sum_{k=1}^K p_k^2$
  where $p_k$ is the proportion of samples belonging to class $k$.
- **Strengths & Weaknesses:**
  - *Strengths:* Captures non-linear relationships and high-order feature interactions automatically; invariant to monotonic feature scaling.
  - *Weaknesses:* A single unpruned decision tree has **high variance** and tends to memorize noisy training samples (overfitting), leading to poor generalization on unseen data.


In [ ]:
# --- Predefined model hyperparameters ---
DT_RANDOM_STATE = 42
DT_MAX_DEPTH = None  # Default unconstrained tree

# YOUR CODE HERE:
# 1. Build a Pipeline combining preprocessor and DecisionTreeClassifier
# dt_pipeline = Pipeline(steps=[
#     (preprocessor, preprocessor),
#     (classifier, DecisionTreeClassifier(random_state=DT_RANDOM_STATE, max_depth=DT_MAX_DEPTH))
# ])

# 2. Fit on (X_train, y_train) and predict on X_val
# dt_pipeline.fit(X_train, y_train)
# y_val_pred_dt = dt_pipeline.predict(X_val)

# 3. Evaluate using evaluate_model and append results to experiments_summary
# dt_results = evaluate_model(y_val, y_val_pred_dt, model_name="Decision Tree (Default)")
# experiments_summary.append(dt_results)


---
## 5. Ensemble Bagging: Random Forest Classifier

### 🎯 Objectives:
1. Build and train a `RandomForestClassifier(n_estimators=100, random_state=42)`.
2. Evaluate ensemble bagging performance on `X_val`.
3. Observe how averaging multiple de-correlated trees stabilizes predictions.

### 💡 Conceptual Explanation & Intuition:
- **What is Bagging (Bootstrap Aggregating)?**
  Instead of relying on a single tree, Random Forest builds an ensemble of $B$ independent decision trees (e.g. $B=100$) and averages their class probability predictions:
  $P(y = k \mid \mathbf{x}) = \frac{1}{B} \sum_{b=1}^B P_b(y = k \mid \mathbf{x})$
- **Two Layers of Randomness (De-correlation):**
  1. **Bootstrap Sample:** Each tree is trained on a random sample of rows drawn with replacement from the training set.
  2. **Feature Subspace Sampling:** At each split, the tree can only select from a random subset of features (typically $\sqrt{D}$). This prevents one dominant feature from dictating every tree.
- **Why Bagging Works:** Individual trees have low bias but high variance. Averaging many de-correlated trees dramatically reduces variance without increasing bias.


In [ ]:
# --- Predefined model hyperparameters ---
RF_N_ESTIMATORS = 100
RF_RANDOM_STATE = 42
RF_N_JOBS = -1

# YOUR CODE HERE:
# 1. Build a Pipeline combining preprocessor and RandomForestClassifier
# rf_pipeline = Pipeline(steps=[
#     (preprocessor, preprocessor),
#     (classifier, RandomForestClassifier(n_estimators=RF_N_ESTIMATORS, random_state=RF_RANDOM_STATE, n_jobs=RF_N_JOBS))
# ])

# 2. Fit on (X_train, y_train) and predict on X_val
# rf_pipeline.fit(X_train, y_train)
# y_val_pred_rf = rf_pipeline.predict(X_val)

# 3. Evaluate using evaluate_model and append results to experiments_summary
# rf_results = evaluate_model(y_val, y_val_pred_rf, model_name="Random Forest (100 Trees)")
# experiments_summary.append(rf_results)


---
## 6. Ensemble Boosting: XGBoost Classifier

### 🎯 Objectives:
1. Build and train an `XGBClassifier` configured for multi-class classification (`objective=multi:softprob`).
2. Evaluate gradient boosting performance on `X_val`.
3. Contrast the boosting paradigm (sequential error correction) against bagging (independent averaging).

### 💡 Conceptual Explanation & Intuition:
- **What is Gradient Boosting?**
  Unlike Random Forest (which trains trees independently in parallel), Gradient Boosting trains trees **sequentially in series**.
  Each new tree $t$ is trained to predict the negative gradient (the residual errors) of the ensemble built so far:
  $\hat{y}^{(t)} = \hat{y}^{(t-1)} + \eta \cdot f_t(\mathbf{x})$
  where $\eta$ is the **learning rate (shrinkage)** and $f_t$ is the new tree.
- **Why XGBoost (Extreme Gradient Boosting) Dominates Tabular Data:**
  - **Second-Order Gradients:** Uses both gradient ($g_i$) and hessian ($h_i$) for faster, more accurate optimization steps.
  - **Regularization:** Penalizes model complexity ($\ell_1$ Lasso, $\ell_2$ Ridge) directly in the tree loss function to combat overfitting:
    $\mathcal{L}_{\text{tree}} = \sum_{j=1}^T \left[ G_j w_j + \frac{1}{2}(H_j + \lambda) w_j^2 \right] + \gamma T$
  - **Sequential Learning:** Trees focus learning effort specifically on difficult examples that earlier trees misclassified.


In [ ]:
# --- Predefined model hyperparameters ---
XGB_PARAMS = {
    "n_estimators": 100,
    "learning_rate": 0.1,
    "max_depth": 4,
    "objective": "multi:softprob",
    "num_class": 5,
    "random_state": 42,
    "eval_metric": "mlogloss"
}

# YOUR CODE HERE:
# 1. Build a Pipeline combining preprocessor and XGBClassifier
# xgb_pipeline = Pipeline(steps=[
#     (preprocessor, preprocessor),
#     (classifier, XGBClassifier(**XGB_PARAMS))
# ])

# 2. Fit on (X_train, y_train) and predict on X_val
# xgb_pipeline.fit(X_train, y_train)
# y_val_pred_xgb = xgb_pipeline.predict(X_val)

# 3. Evaluate using evaluate_model and append results to experiments_summary
# xgb_results = evaluate_model(y_val, y_val_pred_xgb, model_name="XGBoost (Default)")
# experiments_summary.append(xgb_results)


---
## 7. Model Comparison & Benchmark Summary

### 🎯 Objectives:
1. Compile all experiment results into a structured comparison DataFrame.
2. Rank models by our primary metric (**Macro F1**) and secondary metric (**Accuracy**).
3. Visualize benchmark performance using a grouped bar chart.

### 💡 Conceptual Explanation & Intuition:
- **Data-Driven Model Selection:** In machine learning, no single algorithm is guaranteed to perform best on every dataset (the *No Free Lunch Theorem*). Systematically logging validation metrics across linear, tree, and ensemble architectures ensures our selection of the final candidate for hyperparameter tuning is backed by objective empirical evidence.


In [ ]:
# --- Predefined visualization styling and table formatting ---
comparison_columns = ["model", "macro_f1", "accuracy", "weighted_f1"]

# YOUR CODE HERE:
# 1. Convert experiments_summary into a pandas DataFrame sorted by macro_f1 descending
# comparison_df = pd.DataFrame(experiments_summary)[comparison_columns]
# comparison_df = comparison_df.sort_values(by="macro_f1", ascending=False).reset_index(drop=True)
# display(comparison_df)

# 2. Plot a horizontal bar chart comparing Macro F1 across candidate models
# fig, ax = plt.subplots(figsize=(9, 4))
# sns.barplot(data=comparison_df, x="macro_f1", y="model", palette="mako", ax=ax)
# ax.set_title("Model Validation Comparison — Macro F1-Score (Primary Metric)", fontsize=13, fontweight="bold")
# ax.set_xlabel("Macro F1-Score")
# ax.set_ylabel("Model")
# for p in ax.patches:
#     width = p.get_width()
#     ax.annotate(f"{width:.4f}", (width, p.get_y() + p.get_height() / 2.),
#                 ha="left", va="center", fontsize=10, xytext=(5, 0), textcoords="offset points")
# plt.xlim(0, 1.0)
# plt.show()


---
## 8. Summary & Next Steps

### 📝 Key Takeaways from Stage 4 & 5:
1. **Automated Preprocessing:** Built a robust `ColumnTransformer` handling numerical scaling (`StandardScaler`) and categorical encoding (`OneHotEncoder`) without data leakage.
2. **Model Diversity:** Benchmarked 4 distinct model families:
   - **Logistic Regression:** Simple, fast, linear baseline.
   - **Decision Tree:** Non-linear decision rules, high variance.
   - **Random Forest:** Ensemble bagging reducing variance across 100 de-correlated trees.
   - **XGBoost:** Gradient boosting sequentially optimizing residuals with regularized leaf weights.
3. **Primary Benchmark:** Tracked **Macro F1** and **Accuracy** across all models on the identical 20% validation split.

➡️ **Next Stage:** Section 6 — Hyperparameter Tuning with Stratified K-Fold Cross-Validation on the top-performing model family!
